In [ ]:
import pandas as pd
from google import genai
from google.genai import types
import json
import time

# 1. Configuración del Cliente
client = genai.Client(api_key="API_KEY")
MODEL_ID = "gemini-3.1-flash-lite-preview"

# 2. Carga del Dataset Original
df_real = pd.read_excel('parte_6.xlsx')

TECNICAS = [
    "exageración de cifras", "cambio de contexto temporal", "cambio de localización",
    "generalización indebida", "uso de términos ambiguos o emocionales", "titular engañoso",
    "mezcla de dato real con conclusion falsa", "reformulación sesgada de titular", 
    "reinterpretación narrativa de datos reales"
]

def transformar_noticia_individual(row, idx, max_retries=10):
    """
    Envía una única noticia completa. Si falla, reintenta hasta max_retries veces.
    """
    prompt = f"""
    Eres un experto en análisis de desinformación. Transforma esta noticia REAL en FALSA o MISLEADING (contexto engañoso).
    
    INSTRUCCIONES:
    1. Modifica el titulo y el cuerpo (mínimo 1500 caracteres sin espacios) manteniendo el tema.
    2. Usa una de estas técnicas: {TECNICAS}.
    3. Etiqueta: 'FALSO' o 'CONTEXTO'.

    NOTICIA ORIGINAL (ID_{idx}):
    titulo: {row['titulo']}
    texto: {row['texto']}

    Responde en JSON:
    {{
      "id": "ID_{idx}",
      "titulo": "...",
      "texto": "...",
      "etiqueta": "FALSO/CONTEXTO",
      "tecnica": "...",
      "modificacion": "..."
    }}
    """

    for intento in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.8
                )
            )
            
            texto_json = response.text.strip()
            if texto_json.startswith("```"):
                texto_json = texto_json.split("json")[-1].split("```")[0].strip()
                
            return json.loads(texto_json)
            
        except Exception as e:
            espera = (intento + 1) * 5  # Espera 5, 10, 15 segundos...
            print(f"Error en ID_{idx} (Intento {intento+1}/{max_retries}): {e}. Reintentando en {espera}s...")
            time.sleep(espera)
            
    return None

# 3. Bucle de Procesamiento 1 a 1
resultados_acumulados = []
archivo_salida = 'parte_6_falso_misleading.xlsx'

print(f"Iniciando proceso: {len(df_real)} noticias...")

for i in range(len(df_real)):
    fila_actual = df_real.iloc[i]
    print(f"[{i+1}/{len(df_real)}] Procesando ID_{i}...")
    
    # Llamada con reintentos
    resultado = transformar_noticia_individual(fila_actual, i)
    
    if resultado:
        resultados_acumulados.append(resultado)
    else:
        print(f"!! ID_{i} falló tras todos los reintentos. Saltando...")

    # 4. Guardado incremental
    if (i + 1) % 10 == 0 or (i + 1) == len(df_real):
        if resultados_acumulados: # Solo guarda si hay datos nuevos
            df_ai = pd.DataFrame(resultados_acumulados)
            df_ai['id_num'] = df_ai['id'].str.extract(r'(\d+)').astype(int)
            df_ai.set_index('id_num', inplace=True)
            
            df_temp = df_real.loc[df_ai.index].copy()
            
            df_temp['titulo'] = df_ai['titulo']
            df_temp['texto'] = df_ai['texto']
            df_temp['etiqueta'] = df_ai['etiqueta']
            df_temp['tecnica'] = df_ai['tecnica']
            df_temp['modificacion'] = df_ai['modificacion']
            
            columnas_finales = ['titulo', 'etiqueta', 'texto', 'tema', 'tecnica', 'modificacion', 'fecha_publicacion']
            df_export = df_temp[columnas_finales]
            
            df_export.to_excel(archivo_salida, index=False)
            print(f">> Progreso guardado: {len(df_export)} filas.")



print(f"\nGeneración finalizada. Archivo guardado como: {archivo_salida}")

Iniciando proceso: 450 noticias...
[1/450] Procesando ID_0...
[2/450] Procesando ID_1...
[3/450] Procesando ID_2...
[4/450] Procesando ID_3...
[5/450] Procesando ID_4...
[6/450] Procesando ID_5...
[7/450] Procesando ID_6...
[8/450] Procesando ID_7...
[9/450] Procesando ID_8...
[10/450] Procesando ID_9...
>> Progreso guardado: 10 filas.
[11/450] Procesando ID_10...
[12/450] Procesando ID_11...
[13/450] Procesando ID_12...
[14/450] Procesando ID_13...
[15/450] Procesando ID_14...
[16/450] Procesando ID_15...
[17/450] Procesando ID_16...
[18/450] Procesando ID_17...
[19/450] Procesando ID_18...
[20/450] Procesando ID_19...
>> Progreso guardado: 20 filas.
[21/450] Procesando ID_20...
[22/450] Procesando ID_21...
[23/450] Procesando ID_22...
[24/450] Procesando ID_23...
[25/450] Procesando ID_24...
[26/450] Procesando ID_25...
[27/450] Procesando ID_26...
[28/450] Procesando ID_27...
[29/450] Procesando ID_28...
[30/450] Procesando ID_29...
>> Progreso guardado: 30 filas.
[31/450] Procesand